# ArcFace

In [5]:
!pip3 install insightface onnxruntime opencv-python
import cv2, torch, insightface, os
import numpy as np
from insightface.app import FaceAnalysis
from insightface.data import get_image as ins_get_image
from torch.nn.functional import cosine_similarity
from collections import Counter
print("Current working directory:", os.getcwd())
model_dir = os.path.join(os.getcwd(), "models")

app = FaceAnalysis(
    root=model_dir,
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)
app.prepare(ctx_id=-1, det_size=(640, 640))



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Current working directory: g:\.thesis\named-ai\data-preprocessing
download_path: g:\.thesis\named-ai\data-preprocessing\models\models\buffalo_l


100%|██████████| 281857/281857 [00:09<00:00, 30848.55KB/s]


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: g:\.thesis\named-ai\data-preprocessing\models\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: g:\.thesis\named-ai\data-preprocessing\models\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: g:\.thesis\named-ai\data-preprocessing\models\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: g:\.thesis\named-ai\data-preprocessing\models\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: g:\.thesis\named-ai\data-p

In [9]:
print("=== InsightFace Model Info ===")
for key, model in app.models.items():
    print(f"{key:12s} -> {model.model_file}")

=== InsightFace Model Info ===
landmark_3d_68 -> g:\.thesis\named-ai\data-preprocessing\models\models\buffalo_l\1k3d68.onnx
landmark_2d_106 -> g:\.thesis\named-ai\data-preprocessing\models\models\buffalo_l\2d106det.onnx
detection    -> g:\.thesis\named-ai\data-preprocessing\models\models\buffalo_l\det_10g.onnx
genderage    -> g:\.thesis\named-ai\data-preprocessing\models\models\buffalo_l\genderage.onnx
recognition  -> g:\.thesis\named-ai\data-preprocessing\models\models\buffalo_l\w600k_r50.onnx


In [8]:
for name, model in app.models.items():
    path = getattr(model, "model_file", None)
    print(f"{name}: {path}")

landmark_3d_68: g:\.thesis\named-ai\data-preprocessing\models\models\buffalo_l\1k3d68.onnx
landmark_2d_106: g:\.thesis\named-ai\data-preprocessing\models\models\buffalo_l\2d106det.onnx
detection: g:\.thesis\named-ai\data-preprocessing\models\models\buffalo_l\det_10g.onnx
genderage: g:\.thesis\named-ai\data-preprocessing\models\models\buffalo_l\genderage.onnx
recognition: g:\.thesis\named-ai\data-preprocessing\models\models\buffalo_l\w600k_r50.onnx


In [ ]:
def build_facebank(facebank_dir="facebank"):
    names, embeddings = [], []

    for person_name in os.listdir(facebank_dir):
        person_dir = os.path.join(facebank_dir, person_name)
        if not os.path.isdir(person_dir):
            continue

        person_embeddings = []
        for img_name in os.listdir(person_dir):
            img_path = os.path.join(person_dir, img_name)
            img = cv2.imread(img_path)

            if img is None:
                continue

            faces = app.get(img)
            if len(faces) == 0:
                continue

            emb = torch.tensor(faces[0]["embedding"])
            person_embeddings.append(emb)

        if person_embeddings:
            # Average embedding for that person
            mean_emb = torch.stack(person_embeddings).mean(dim=0)
            mean_emb = torch.nn.functional.normalize(mean_emb, p=2, dim=0)

            names.append(person_name) #y
            embeddings.append(mean_emb) #x

    if not embeddings:
        raise ValueError("No embeddings found in facebank.")

    facebank = torch.stack(embeddings)
    print(f"Built facebank with {len(names)} identities.")
    return names, facebank

In [4]:
def save_facebank(names, embeddings, path="facebank.pt"):
    torch.save({"names": names, "embeddings": embeddings}, path)
    print(f"Facebank saved to {path}")

In [8]:
def load_facebank(path="facebank.pt"):
    data = torch.load(path)
    print(f"Loaded facebank with {len(data['names'])} identities.")
    return data["names"], data["embeddings"]

In [12]:
def recognize_face(img_path, facebank_names, facebank_embeddings, k=3, threshold=0.5):
    img = cv2.imread(img_path)
    faces = app.get(img)

    if len(faces) == 0:
        return "No face detected", 0.0

    test_emb = np.array(faces[0]['embedding'])
    test_emb = test_emb / np.linalg.norm(test_emb)

    if isinstance(facebank_embeddings, torch.Tensor):
        facebank_embeddings = facebank_embeddings.cpu().numpy()

    facebank_embeddings = facebank_embeddings / np.linalg.norm(facebank_embeddings, axis=1, keepdims=True)
    sims = np.dot(facebank_embeddings, test_emb)

    # --- Debug info: see top-K neighbors ---
    topk_idxs = np.argsort(sims)[-k:][::-1]
    print(f"Top-{k} similarities:")
    for i in topk_idxs:
        print(f"  {facebank_names[i]}: {sims[i]:.4f}")

    topk_labels = [facebank_names[i] for i in topk_idxs]
    topk_sims = sims[topk_idxs]

    from collections import Counter
    best_label = Counter(topk_labels).most_common(1)[0][0]
    confidence = np.mean([s for s, l in zip(topk_sims, topk_labels) if l == best_label])

    print(f"\nPredicted (majority vote): {best_label} with avg similarity {confidence:.4f}\n")

    if confidence < threshold:
        return "Unknown", confidence
    else:
        return best_label, confidence


In [ ]:
print("Current working directory:", os.getcwd())
names, embeddings = build_facebank()
save_facebank(names, embeddings)

Current working directory: g:\.thesis\named-ai\data-preprocessing


In [ ]:
names, embeddings = load_facebank() #loads facebank from facebank.pt 
identity, confidence = recognize_face("test_images/Andy Samberg_91.jpg", names, embeddings)
#print(f"Predicted: {identity} (Confidence: {confidence:.3f})")
identity, confidence = recognize_face("test_images/Akshay Kumar_45.jpg", names, embeddings)
#print(f"Predicted: {identity} (Confidence: {confidence:.3f})")
identity, confidence = recognize_face("test_images/Akshay Kumar_46.jpg", names, embeddings)
#print(f"Predicted: {identity} (Confidence: {confidence:.3f})")
print(embeddings)

Loaded facebank with 31 identities.
Top-3 similarities:
  Andy Samberg: 0.8069
  Camila Cabello: 0.1197
  Lisa Kudrow: 0.0694

Predicted (majority vote): Andy Samberg with avg similarity 0.8069

Top-3 similarities:
  Akshay Kumar: 0.7804
  Hrithik Roshan: 0.1338
  Lisa Kudrow: 0.1062

Predicted (majority vote): Akshay Kumar with avg similarity 0.7804

Top-3 similarities:
  Akshay Kumar: 0.7034
  Hrithik Roshan: 0.2850
  Anushka Sharma: 0.1267

Predicted (majority vote): Akshay Kumar with avg similarity 0.7034

tensor([[ 1.4116e-02, -2.2324e-02,  3.6334e-02,  ...,  4.7298e-03,
          2.4590e-02, -3.2044e-02],
        [-5.8645e-02,  1.0540e-02,  2.1642e-02,  ...,  6.3522e-02,
         -4.0187e-03,  2.4414e-02],
        [ 2.5552e-03,  1.8102e-02,  2.2906e-02,  ...,  7.8302e-02,
          4.3908e-02, -4.4635e-03],
        ...,
        [ 1.1268e-02, -1.1289e-02, -1.6800e-02,  ...,  1.6786e-02,
         -3.5777e-02, -4.8672e-02],
        [ 8.1160e-06, -1.1532e-02,  5.1288e-02,  ..., -8.70

: 

asdasd